# Cellpose GRAM-W v4 — training and evaluation on Kaggle

**Before running:** right panel → Accelerator **GPU T4 x2**, Internet **On**, and **+ Add Input** → your dataset.
For the long run use **Save Version → Save & Run All (Commit)**; results appear in the **Output** tab.

Expected dataset content (Kaggle may unpack the `.zip` and model files into folders; the code handles both):
```
data_0.26/train/   data_0.26/test/   split.csv
test_originals/    (the 14 test images + their _ROI.zip)
models/GRAMW_v3    (and optionally GRAMW_v2)
```

## 0. Install

In [ ]:
!pip install -q "cellpose>=4,<5" roifile

## 1. Setup

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import torch
from skimage.transform import rescale, resize
from skimage.draw import polygon
from skimage.segmentation import clear_border, relabel_sequential
from roifile import roiread
from cellpose import models, utils, train, metrics, io
io.logger_setup()

# ---- settings (must match the Colab notebook) ----
GRAM           = 0
TARGET         = 0.26
EDGE_BUFFER_UM = 0.8

# ---- locate the dataset: wherever split.csv is ----
DATA = list(Path("/kaggle/input").rglob("split.csv"))[0].parent
WORK = Path("/kaggle/working")
train_dir = DATA / f"data_{TARGET:.2f}" / "train"
test_dir  = DATA / f"data_{TARGET:.2f}" / "test"
orig_dir  = DATA / "test_originals"
model_in  = DATA / "models"
model_out = WORK / "models"
model_out.mkdir(exist_ok=True)
for p in (train_dir, test_dir, orig_dir, model_in):
    print(f"{'OK     ' if p.exists() else 'MISSING'} {p}")

split = pd.read_csv(DATA / "split.csv")

# ---- images ----
def pixel_size(path):
    with tifffile.TiffFile(path) as tif:
        xres = tif.pages[0].tags.get("XResolution")
    if xres is None:
        return None
    pixels, length = xres.value
    return length / pixels

def get_gram(img):
    if img.ndim == 2:
        return img
    if img.ndim == 3:
        return img[GRAM]
    raise ValueError(f"Unexpected image shape {img.shape}")

def load_set(d):
    X, Y = [], []
    for mp in sorted(d.glob("*_masks.tif")):
        X.append(tifffile.imread(mp.with_name(mp.name.replace("_masks", ""))))
        Y.append(tifffile.imread(mp).astype(np.int32))
    return X, Y

# ---- ROIs: works whether Kaggle kept the _ROI.zip or unpacked it into a folder of .roi files ----
def find_roi(stem):
    for name in (stem + "_ROI.zip", stem + "_ROI", stem + "_ROI.roi"):
        p = orig_dir / name
        if p.exists():
            return p
    raise FileNotFoundError(f"No ROI file or folder for {stem}")

def read_rois(path):
    if path.is_dir():
        rois = []
        for p in sorted(path.rglob("*.roi")):
            r = roiread(p)
            if not r.name:
                r.name = p.stem
            rois.append(r)
        return rois
    rois = roiread(path)
    return rois if isinstance(rois, list) else [rois]

def rois_to_mask(rois, shape):
    mask = np.zeros(shape, dtype=np.int32)
    for n, roi in enumerate(rois, start=1):
        xy = roi.coordinates()
        rr, cc = polygon(xy[:, 1] - 0.5, xy[:, 0] - 0.5, shape)
        mask[rr, cc] = n
    return mask

def roi_tag(name):
    name = (name or "").lower()
    if name.startswith("abn_"):
        return "abnormal"
    if name.startswith("unsure_"):
        return "unsure"
    return "normal"

def remove_edge(m, buffer_px):
    return relabel_sequential(clear_border(m, buffer_size=buffer_px))[0]

def edge_buffer_px(pixel_um):
    return max(1, round(EDGE_BUFFER_UM / pixel_um))

def drop_on(mask_pred, region):
    """Remove predicted cells that lie mostly (>50%) inside `region`."""
    out = mask_pred.copy()
    for lab in range(1, mask_pred.max() + 1):
        cell = mask_pred == lab
        if cell.any() and region[cell].mean() > 0.5:
            out[cell] = 0
    return relabel_sequential(out)[0]

# ---- models: repack a model that Kaggle unpacked into a folder ----
def model_file(name):
    """Path to a usable model file: trained here, uploaded as a file, or uploaded and unpacked by Kaggle."""
    for p in (model_out / name, model_in / name):
        if p.is_file():
            return p
    src = model_in / name
    if src.is_dir():
        pkl = next(src.rglob("data.pkl"))
        root = pkl.parent
        packed = model_out / f"{name}_repacked"
        with zipfile.ZipFile(packed, "w", compression=zipfile.ZIP_STORED) as z:
            for p in sorted(root.rglob("*")):
                if p.is_file():
                    z.write(p, arcname=f"{root.name}/{p.relative_to(root).as_posix()}")
        return packed
    return None

def load_model(name):
    if name == "pretrained":
        return models.CellposeModel(gpu=True)
    return models.CellposeModel(gpu=True, pretrained_model=str(model_file(name)))

def predict_full(m, path, target=TARGET):
    gram_full = get_gram(tifffile.imread(path))
    small = rescale(gram_full, pixel_size(path) / target, order=1, preserve_range=True, anti_aliasing=True)
    pred = m.eval(small)[0]
    return resize(pred, gram_full.shape, order=0, preserve_range=True, anti_aliasing=False).astype(np.int32)

def show_outlines(image, masks, ax, color="red", title=""):
    lo, hi = np.percentile(image, [1, 99.5])
    ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)
    for o in utils.outlines_list(masks):
        ax.plot(o[:, 0], o[:, 1], color=color, linewidth=1)
    ax.set_title(title)
    ax.axis("off")

print(split.groupby(["split", "mode"]).size())

## 2. Train v4

Set `N_EPOCHS = 1` for a quick test first if you like, then back to 25 before the committed run.

In [ ]:
MODEL_NAME    = "GRAMW_v4"
N_EPOCHS      = 25
LEARNING_RATE = 1e-5

X_train, Y_train = load_set(train_dir)
X_test,  Y_test  = load_set(test_dir)
print(len(X_train), "train images,", len(X_test), "test images")

ft_model = models.CellposeModel(gpu=True)
model_path, train_losses, test_losses = train.train_seg(
    ft_model.net,
    train_data=X_train, train_labels=Y_train,
    test_data=X_test,   test_labels=Y_test,
    learning_rate=LEARNING_RATE, weight_decay=0.1,
    n_epochs=N_EPOCHS, batch_size=1, min_train_masks=1,
    save_every=5,
    save_path=WORK, model_name=MODEL_NAME,
)
(model_out / f"{MODEL_NAME}.json").write_text(json.dumps(
    {"TARGET": TARGET, "epochs": N_EPOCHS, "learning_rate": LEARNING_RATE, "n_train": len(X_train)}))
print("saved to", model_path)

del ft_model
torch.cuda.empty_cache()

In [ ]:
tl = np.array(test_losses)
ep = np.nonzero(tl)[0]
plt.plot(train_losses, label="train")
plt.plot(ep, tl[ep], "o-", label="test")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.savefig(WORK / f"{MODEL_NAME}_loss.png", dpi=150)
plt.show()

## 3. Evaluate on the 14 test images (full resolution, edge cells removed)

- **all cells:** every ROI counts (normal, `abn_`, `unsure_`).
- **normal cells:** only normal ROIs count; detections on `abn_`/`unsure_` cells are ignored.

In [ ]:
COMPARE = ["pretrained", "GRAMW_v2", "GRAMW_v3", MODEL_NAME]

names = list(split.loc[split["split"] == "test", "name"])
files = []
for n in names:
    hits = [p for p in orig_dir.glob(n + ".tif*") if p.is_file()]
    if hits:
        files.append(hits[0])
    else:
        print("⚠ original image not found for", n)
modes = ["SR" if pixel_size(f) < 0.09 else "non-SR" for f in files]

truth_all, truth_normal, ignore, images = [], [], [], []
for f in files:
    gram_full = get_gram(tifffile.imread(f))
    rois = read_rois(find_roi(f.stem))
    full = clear_border(rois_to_mask(rois, gram_full.shape), buffer_size=edge_buffer_px(pixel_size(f)))
    normal_labels = [n for n, r in enumerate(rois, start=1) if roi_tag(r.name) == "normal"]
    is_normal = np.isin(full, normal_labels)
    truth_all.append(relabel_sequential(full)[0])
    truth_normal.append(relabel_sequential(np.where(is_normal, full, 0))[0])
    ignore.append((full > 0) & ~is_normal)
    images.append(gram_full)

results = {}
for name in COMPARE:
    if name != "pretrained" and model_file(name) is None:
        print(f"⚠ skipping {name}: not found")
        continue
    m = load_model(name)
    print(f"segmenting with {name}…", flush=True)
    results[name] = [remove_edge(predict_full(m, f), edge_buffer_px(pixel_size(f))) for f in files]
    del m
    torch.cuda.empty_cache()

def score_line(truth, preds):
    ap, tp, fp, fn = metrics.average_precision(truth, preds, threshold=[0.5, 0.75, 0.9])
    a = np.nanmean(ap, axis=0)
    return (f"AP@0.5={a[0]:.3f}  AP@0.75={a[1]:.3f}  AP@0.9={a[2]:.3f}"
            f"  | TP={tp.sum(0)[0]:.0f}  FP={fp.sum(0)[0]:.0f}  FN={fn.sum(0)[0]:.0f}")

lines = []
for kind in ["all cells", "normal cells"]:
    for group in ["all", "SR", "non-SR"]:
        idx = [k for k, md_ in enumerate(modes) if group == "all" or md_ == group]
        lines.append(f"--- {kind} | {group} images ({len(idx)}) ---")
        for name, preds in results.items():
            if kind == "all cells":
                t_, p_ = [truth_all[k] for k in idx], [preds[k] for k in idx]
            else:
                t_, p_ = [truth_normal[k] for k in idx], [drop_on(preds[k], ignore[k]) for k in idx]
            lines.append(f"{name:14s} " + score_line(t_, p_))
print("\n".join(lines))
(WORK / f"{MODEL_NAME}_scores.txt").write_text("\n".join(lines))

In [ ]:
for k, f in enumerate(files):
    fig, axs = plt.subplots(1, len(results) + 1, figsize=(6 * (len(results) + 1), 6))
    show_outlines(images[k], truth_normal[k], axs[0], color="lime")
    for o in utils.outlines_list(np.where(ignore[k], truth_all[k], 0)):
        axs[0].plot(o[:, 0], o[:, 1], color="orange", linewidth=1.5)
    axs[0].set_title(f"{modes[k]}: your ROIs (abn_/unsure_ orange)")
    for ax, name in zip(axs[1:], results):
        show_outlines(images[k], results[name][k], ax, title=name)
    plt.show()

## 4. Clean up before downloading

Repacked copies of uploaded models are only needed inside this run; deleting them keeps the Output small.

In [ ]:
for p in model_out.glob("*_repacked"):
    p.unlink()
print(sorted(p.name for p in model_out.iterdir()))